# Chapter 03: Pivot Tables

Pivot tables reshape data from long (tidy) format into wide format, making it easier to spot patterns across two dimensions. They are one of the most commonly used tools in spreadsheet software, and pandas provides a powerful programmatic equivalent.

In this notebook we will cover:

- `.pivot()` vs `.pivot_table()`
- `pivot_table()` with `values`, `index`, `columns`, `aggfunc`
- The `margins` parameter
- `.melt()` for unpivoting (wide to long)
- `.stack()` and `.unstack()`
- Reference diagram: `reshaping_pivot.png`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
tips = pd.read_csv('data/tips.csv')
tips.head()

## Reshaping Concept

The following diagram illustrates how pivot and melt operations reshape data:

![Reshaping Pivot](data/reshaping_pivot.png)

## `.pivot()` — Simple Reshaping (No Aggregation)

`.pivot()` reshapes data from long to wide format. It requires that the combination of `index` and `columns` values is unique — if there are duplicates, it will raise an error.

In [ ]:
# Create a simple example where each (date, city) pair is unique
weather = pd.DataFrame({
    'date': ['2024-01-01', '2024-01-01', '2024-01-02', '2024-01-02'],
    'city': ['Sydney', 'Melbourne', 'Sydney', 'Melbourne'],
    'temperature': [32, 28, 30, 25]
})
weather

In [ ]:
# Pivot: dates as rows, cities as columns
weather.pivot(index='date', columns='city', values='temperature')

## `.pivot_table()` — Pivot with Aggregation

When duplicate (index, column) pairs exist, you need `pivot_table()`, which applies an aggregation function to combine them. This is the pandas equivalent of Excel's PivotTable.

Key parameters:
- `values`: column(s) to aggregate
- `index`: column(s) to use as row labels
- `columns`: column(s) to use as column headers
- `aggfunc`: aggregation function (default is `'mean'`)
- `margins`: add row/column totals

In [ ]:
# Average total_bill by day and time
pd.pivot_table(tips, values='total_bill', index='day', columns='time', aggfunc='mean')

In [ ]:
# Count of meals by day and smoker status
pd.pivot_table(tips, values='total_bill', index='day', columns='smoker', aggfunc='count')

In [ ]:
# Sum of tips by day and sex
pd.pivot_table(tips, values='tip', index='day', columns='sex', aggfunc='sum')

In [ ]:
# Multiple aggregation functions
pd.pivot_table(tips, values='total_bill', index='day', columns='time',
               aggfunc=['mean', 'count', 'sum'])

In [ ]:
# Multiple values columns
pd.pivot_table(tips, values=['total_bill', 'tip'], index='day',
               columns='time', aggfunc='mean')

## The `margins` Parameter

Setting `margins=True` adds an "All" row and column showing the overall aggregation — like totals in a spreadsheet.

In [ ]:
pd.pivot_table(tips, values='total_bill', index='day', columns='time',
               aggfunc='mean', margins=True)

In [ ]:
pd.pivot_table(tips, values='tip', index='day', columns='smoker',
               aggfunc='sum', margins=True)

## Pivot Table with the Sales Funnel Dataset

In [ ]:
funnel = pd.read_csv('data/Sales_Funnel_CRM.csv')
funnel.head()

In [ ]:
# Total sale price by company and product
pd.pivot_table(funnel, values='Sale Price', index='Company',
               columns='Product', aggfunc='sum')

In [ ]:
# Average sale price by status and account manager
pd.pivot_table(funnel, values='Sale Price', index='Account Manager',
               columns='Status', aggfunc='mean', margins=True)

## `.melt()` — Unpivoting (Wide to Long)

`.melt()` is the inverse of pivoting. It converts a wide-format DataFrame into a long-format one, which is often needed for plotting libraries and tidy data analysis.

In [ ]:
# Start with a wide-format DataFrame
wide = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Charlie'],
    'maths': [90, 85, 78],
    'science': [88, 92, 75],
    'english': [95, 80, 82]
})
wide

In [ ]:
# Melt into long format
long = wide.melt(id_vars='student', var_name='subject', value_name='score')
long

In [ ]:
# You can pivot it back to wide format
long.pivot(index='student', columns='subject', values='score')

In [ ]:
# Melt only specific columns
wide.melt(id_vars='student', value_vars=['maths', 'science'],
          var_name='subject', value_name='score')

## `.stack()` and `.unstack()`

These methods work on the index/columns directly:

- `.stack()`: moves columns into a new level of the row index (wide to long)
- `.unstack()`: moves a row index level into columns (long to wide)

In [ ]:
# Create a pivot table as our starting point
pt = pd.pivot_table(tips, values='total_bill', index='day', columns='time', aggfunc='mean')
pt

In [ ]:
# Stack: move columns (time) into the index
stacked = pt.stack()
stacked

In [ ]:
# Unstack: move the inner index level back to columns
stacked.unstack()

In [ ]:
# Unstack a different level
stacked.unstack(level=0)

In [ ]:
# GroupBy + unstack is a common pattern (alternative to pivot_table)
tips.groupby(['day', 'time'])['total_bill'].mean().unstack()

## Summary

In this notebook we covered pivot tables and reshaping:

- `.pivot()` reshapes when index/column pairs are unique (no aggregation).
- `.pivot_table()` handles duplicates by applying an `aggfunc` (default: mean).
- `margins=True` adds grand totals to rows and columns.
- `.melt()` converts wide format to long format (the inverse of pivoting).
- `.stack()` moves columns into the index; `.unstack()` moves index levels into columns.
- `groupby().unstack()` is often equivalent to `pivot_table()`.

This concludes the Pandas chapter. You now have the tools to load, clean, transform, aggregate, and reshape tabular data in Python.